# Composite liveability index
## Mexicali ULI — `WP00_composite_index`

**Lead:** Carl Higgs
**Schema version:** 1.0.0

Not an analyst task. This notebook is the **reporting
step**: it ingests validated work package deliverables and
turns values-for-places into statements about *people* —
what share of the population experiences what, and a
population-weighted liveability score.

Analysts do not need anything here. Choosing a population
denominator is a decision made once, in one place, after
delivery; urban fabric and exposure measures are properties
of place and are computed as such.

## Population denominators

Four denominators are carried on every reporting geography,
because the right one depends on the question being asked.

| Basis | What it is | Study extent | Condesa |
|---|---|---|---|
| `pop_census_2020` | INEGI census | 852,506 | 0 |
| `population` | GHS-POP 2025 (observed today) | 823,260 | 372 |
| `population_2030` | GHS-POP 2030 (projection as published) | 842,287 | 377 |
| `population_planned` | Condesa at full occupancy | 48,170 | 48,032 |
| `population_scenario_2030` | GHS-POP 2030 outside Condesa **+** planned occupancy inside | **890,079** | 48,032 |

### Why the scenario basis exists

Condesa is platted and roaded — 43 km of street network
across 27 of 40 fraccionamientos — but essentially unbuilt.
No population product resolves it, and the 2030 projection
does not either: it adds **19,760 people city-wide (+2.3%)
and 5 to Condesa**, leaving its share of the city unchanged
at 0.044%. GHS-POP allocates a projected total onto observed
built-up surface, and the development has roads but few
roofs.

What population it does record is borrowed from next door:
364 of the 372 persons sit in fraccionamientos within 100 m
of an already-populated 2020 manzana, and none beyond 500 m.
That is the signature of areal apportionment, not occupancy.

So `population_planned` is a **declared assumption, not an
estimate**: one household per residential lot (14,597 lots
of 14,989 at ≤ 300 m²) at the assumed Mexicali mean
household size of 3.3. Both parameters live in
`uli.geography` and any result derived from them can be
re-derived under different assumptions.

Under that assumption Condesa becomes **48,170 residents —
5.4% of Mexicali**, against the 377 the projection sees.
That is the whole reason the scenario basis is worth having.

> **Report scenario results as conditional.** They answer
> "if Condesa is fully occupied by 2030, on top of the
> projected city, what will people experience?" — not
> "what do people experience?".

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

In [ ]:
denominators = [
    'pop_census_2020', 'population', 'population_2030',
    'population_planned', 'population_scenario_2030',
]
pd.DataFrame([
    dict(
        level=level,
        units=len(uli.geography.units(level)),
        **{
            d: round(uli.geography.units(level)[d].sum())
            for d in denominators
        },
    )
    for level in uli.vocab.GEO_RESOLUTION_ORDER
])

Note that AGEB and manzana capture only ~39,500 and ~37,200
of the 48,170 planned residents: the 2020 census geographies
do not extend over the whole development. **Report scenario
statistics on the grids or the Condesa layers**, not on
census geography.

---
## Ingest the delivered indicators

In [ ]:
delivered, catalogue = uli.collect()
print(f'{len(delivered):,} rows, '
      f'{catalogue["measure_id"].nunique() if len(catalogue) else 0} '
      'measures')
catalogue

---
## Population exposure

"What share of the population has X?" — computed for the
study area, for Condesa, and for the rest of the city, under
whichever denominator the question calls for.

Population with **no value** is reported separately rather
than counted as unexposed: an indicator that does not reach
an area is not evidence that the area is fine.

In [ ]:
MEASURE = catalogue['measure_id'].iloc[0]   # choose one
THRESHOLD = 1.0

for basis in ('population', 'population_scenario_2030'):
    out = uli.exposed_share(
        delivered, MEASURE, threshold=THRESHOLD,
        comparison='at_or_above', geo_level='grid_100m',
        population=basis)
    out.insert(0, 'basis', basis)
    display(out[['basis', 'group', 'population_with_value',
                 'population_meeting', 'share_meeting']])

The contrast between the two bases is the point of the
exercise. Under the observed denominator Condesa is a few
hundred people and its conditions are invisible in the city
total; under the scenario it is 5.4% of Mexicali and pulls
the city-wide figure with it.

In [ ]:
# Population-weighted distribution of a measure
uli.weighted_summary(
    delivered, MEASURE, geo_level='grid_100m',
    population='population_scenario_2030')

---
## Provisional liveability score

**How the composite index is actually calculated is a later
decision.** What follows is deliberately simple — equal
weights, min-max normalisation, mean of available measures —
so that the plumbing can be exercised end to end now, and so
that swapping in the real method is a one-argument change.

Two rules are already enforced, because they are not
matters of taste:

- only measures flagged `include_in_index` are used;
- measures that are **replicated** from a coarser scale are
  excluded at that scale, because a constant contributes
  nothing but noise to a within-city comparison.

In [ ]:
unit_scores, summary = uli.score(
    delivered, catalogue,
    geo_level='grid_100m',
    population='population_scenario_2030',
    # weights={'measure_id': 0.4, ...},   # later decision
    # normalise=my_normalisation,          # later decision
)
print('used:   ', summary.attrs['measures_used'])
print('skipped:', summary.attrs['measures_skipped'])
summary

In [ ]:
# Map it, with Condesa outlined
grid = uli.geography.load('grid_100m').merge(
    unit_scores, on='geo_id', how='left')
ax = grid.plot(column='score', legend=True, figsize=(12, 7),
               missing_kwds={'color': 'lightgrey'})
uli.geography.load('condesa_fraccionamiento').boundary.plot(
    ax=ax, color='crimson', linewidth=1)
ax.set_title('Provisional liveability score, 100 m grid '
             '(Condesa outlined)')
ax.set_axis_off()

---
## Export

One geopackage, one layer per reporting geography, wide
format — the shape QGIS and Reimagina Urbana both want.

In [ ]:
# uli.to_geopackage(delivered, 'outputs/mexicali_uli.gpkg')
# unit_scores.to_csv('outputs/provisional_score_grid_100m.csv',
#                    index=False)